# EQL-Metriken — Gesamtauswertung

Dieses Notebook übernimmt die Rolle, die vorher `test_eql_readability.py`
unter pytest hatte: es baut alle fünf Evaluationsqueries auf, wendet die
Metriken aus `eql_metrics.py` an, prüft dieselben Eigenschaften wie vorher
per `assert` (schlägt die Zelle fehl, ist etwas kaputt — genau wie ein
fehlgeschlagener Test), und druckt am Ende alle Tabellen für die Thesis.

Anders als `eql_demo.ipynb` geht es hier nicht um das Verstehen einzelner
Queries, sondern um die Gesamtauswertung: alle fünf Fälle + das
einleitende PickUpAction-Beispiel in einem Durchlauf, plus Erweiterungen
gegenüber der ursprünglichen Auswertung (handgeschriebenes SQL, symmetrische
Domain-Transparency, Feature-Tabelle, 5-Punkte-Skalierung, Unentschieden-Check).

**Wichtig:** `eql_metrics.py` enthält nur die reinen Metrik-Funktionen und
Konstanten (keine pytest-Tests, keine Session-Fixtures) — die
Test-/Auswertungslogik, die vorher in den `test_*`-Funktionen von
`test_eql_readability.py` steckte, lebt jetzt hier in diesem Notebook.

## Setup

In [1]:
import sys
sys.path.insert(0, "/home/jovyan/krrood/test")  # Pfad ggf. anpassen, siehe Dockerfile

%run ./eql_data_setup.ipynb

from krrood.entity_query_language.factories import entity, variable, an, set_of
from krrood.ormatic.eql_interface import eql_to_sql
import eql_metrics as m

from test.krrood_test.dataset.semantic_world_like_classes import Body, MoveAction

session = create_demo_session()
n_bodies = build_bodies_for_simple_filter(session)
print(f"Setup abgeschlossen. {n_bodies} Bodies angelegt.")

ormatic_interface.py generiert (über einen echten pytest --collect-only-Lauf).
create_demo_session() definiert.
build_bodies_for_simple_filter() definiert.
build_container_handle_world(), build_four_variable_world(), build_container_chain() definiert.
OK: 5 Bodies angelegt.
OK: 3-Variablen-Welt mit 2 Bodies, 1 Connection.
OK: 4-Variablen-Welt mit 4 Bodies, 2 Connections.
OK: Kette mit 4 Containern, 3 Connections.
Setup abgeschlossen. 5 Bodies angelegt.


## 1. Alle fünf Queries aufbauen und übersetzen

Entspricht `test_summary()` aus der alten pytest-Datei: jede Query wird
gebaut, übersetzt, und mit `eql_metrics.analyze()` inklusive
handgeschriebenem SQL ausgewertet.

In [2]:
queries = []

# Gemessen werden die EQL-Quelltexte aus m.EQL_SOURCES — jeweils INKLUSIVE der
# variable(...)-Deklarationen, damit EQL nicht kürzer gerechnet wird, als der
# Code, den man tatsächlich schreiben muss.

# Query 1: Simple filter
b = variable(type_=Body, domain=[])
t1 = eql_to_sql(an(entity(b).where(b.size > 5)), session)
queries.append(m.analyze(
    "Simple filter",
    m.EQL_SOURCES["Simple filter"],
    str(t1.sql_query),
    handwritten_sql=m.HANDWRITTEN_SQL["Simple filter"],
))

# Query 2: Transitive attributes
# Vier Spalten, deckungsgleich mit der Query in eql_demo.ipynb — dort bewusst
# um robot_y/approach_direction erweitert, siehe die Erklärung dort.
move = variable(type_=MoveAction, domain=[])
t2 = eql_to_sql(an(set_of(
    move.robot_x, move.robot_y,
    move.grasp_config.rotate_gripper, move.grasp_config.approach_direction,
)), session)
queries.append(m.analyze(
    "Transitive attrs",
    m.EQL_SOURCES["Transitive attrs"],
    str(t2.sql_query),
    handwritten_sql=m.HANDWRITTEN_SQL["Transitive attrs"],
))

# Query 3: Multi-variable join (3)
world_3 = build_container_handle_world()
C = variable(Container, domain=world_3.bodies)
H = variable(Handle, domain=world_3.bodies)
FC = variable(FixedConnection, domain=world_3.connections)
t3 = eql_to_sql(an(set_of(C, H, FC).where(C == FC.parent, H == FC.child)), session)
queries.append(m.analyze(
    "Multi-join (3 vars)",
    m.EQL_SOURCES["Multi-join (3 vars)"],
    str(t3.sql_query),
    handwritten_sql=m.HANDWRITTEN_SQL["Multi-join (3 vars)"],
))

# Query 4: Multi-variable join (4) — scaling anchor
world_4 = build_four_variable_world()
C2 = variable(Container, domain=world_4.bodies)
H2 = variable(Handle, domain=world_4.bodies)
FC2 = variable(FixedConnection, domain=world_4.connections)
PC2 = variable(PrismaticConnection, domain=world_4.connections)
t4 = eql_to_sql(an(set_of(C2, H2, FC2, PC2).where(
    C2 == FC2.parent, H2 == FC2.child, C2 == PC2.child,
)), session)
queries.append(m.analyze(
    "Multi-join (4 vars)",
    m.EQL_SOURCES["Multi-join (4 vars)"],
    str(t4.sql_query),
    handwritten_sql=m.HANDWRITTEN_SQL["Multi-join (4 vars)"],
))

# Query 5: Complex combined
t5 = eql_to_sql(
    an(entity(b).where(b.size > 5).ordered_by(b.size).limit(10)), session
)
queries.append(m.analyze(
    "Complex combined",
    m.EQL_SOURCES["Complex combined"],
    str(t5.sql_query),
    handwritten_sql=m.HANDWRITTEN_SQL["Complex combined"],
))

# Einleitendes PickUpAction-Beispiel
queries.append(m.analyze(
    "PickUpAction-Beispiel",
    m.EQL_SOURCES["PickUpAction-Beispiel"],
    m.PICKUP_EXAMPLE_TRANSLATOR_SQL,
    handwritten_sql=m.PICKUP_EXAMPLE_HANDWRITTEN_SQL,
))

print(f"{len(queries)} Queries aufgebaut und ausgewertet.")

6 Queries aufgebaut und ausgewertet.


/tmp/ipykernel_154/2357980348.py:41: SAWarning: An alias is being generated automatically against joined entity Mapper[FixedConnectionDAO(FixedConnectionDAO)] due to overlapping tables.  This is a legacy pattern which may be deprecated in a later release.  Use the aliased(<entity>, flat=True) construct explicitly, see the linked example. (Background on this warning at: https://sqlalche.me/e/20/xaj2)
  str(t3.sql_query),
/tmp/ipykernel_154/2357980348.py:41: SAWarning: An alias is being generated automatically against joined entity Mapper[HandleDAO(HandleDAO)] due to overlapping tables.  This is a legacy pattern which may be deprecated in a later release.  Use the aliased(<entity>, flat=True) construct explicitly, see the linked example. (Background on this warning at: https://sqlalche.me/e/20/xaj2)
  str(t3.sql_query),
/tmp/ipykernel_154/2357980348.py:57: SAWarning: An alias is being generated automatically against joined entity Mapper[FixedConnectionDAO(FixedConnectionDAO)] due to over

## 2. Regressions-Checks

Dieselben Eigenschaften, die vorher als pytest-Asserts liefen. Schlägt eine
Zeile fehl, ist etwas an der Übersetzung oder an den Metriken kaputt
gegangen — genauso, wie ein roter Test es vorher angezeigt hätte.

In [3]:
for r in queries:
    assert r['eql']['fk_error_opportunities'] == 0, \
        f'EQL sollte nie FK-Fehler ermöglichen — {r["name"]}'
    assert r['sql']['nesting_depth'] <= r['eql']['nesting_depth'], \
        f'SQL sollte flacher sein (Design-Entscheidung) — {r["name"]}'
    assert r['eql']['extrinsic_load'] <= r['sql']['extrinsic_load'], \
        f'EQL sollte nicht mehr extrinsische Last haben — {r["name"]}'
    assert r['eql']['cognitive_complexity'] <= r['sql']['cognitive_complexity'], \
        f'EQL sollte keine höhere kognitive Komplexität haben — {r["name"]}'

# Die fünf Kern-Evaluationsqueries sollten kürzer als ihr generiertes SQL sein
# (das PickUpAction-Beispiel ist Sonderfall und wird separat betrachtet, siehe unten)
for r in queries:
    if r['name'] == 'PickUpAction-Beispiel':
        continue
    assert r['eql']['char_length'] < r['sql']['char_length'], \
        f'EQL sollte kürzer als Übersetzer-SQL sein — {r["name"]}'

print("Alle Regressions-Checks bestanden.")

Alle Regressions-Checks bestanden.


## 3. Unentschieden explizit prüfen

Statt nur einseitig `<=` zu behaupten: hier wird aktiv nach Query-Paaren mit
gleicher kognitiver Komplexität gesucht und ausgegeben, statt sie stillschweigend
zuzulassen.

In [4]:
print("Ties in kognitiver Komplexität:")
found_tie = False
for i, r1 in enumerate(queries):
    for r2 in queries[i + 1:]:
        if r1['eql']['cognitive_complexity'] == r2['eql']['cognitive_complexity']:
            found_tie = True
            print(f'  {r1["name"]} und {r2["name"]} haben beide EQL CC = {r1["eql"]["cognitive_complexity"]}')
if not found_tie:
    print("  (keine in diesem Durchlauf)")

Ties in kognitiver Komplexität:
  Simple filter und Complex combined haben beide EQL CC = 3
  Transitive attrs und PickUpAction-Beispiel haben beide EQL CC = 1


## 4. Tabelle 1 — Komplexitätsmetriken

**Begriffe:** `EQL` = EQL-Quelltext, `GenSQL` = vom Übersetzer generiertes
SQL, `Hand` = handgeschriebenes SQL (Referenz, siehe `HANDWRITTEN_SQL`).
Jede Metrik wird als eigene kleine Tabelle mit allen drei Varianten
nebeneinander gezeigt, damit EQL durchgängig sowohl gegen das generierte als
auch gegen das handgeschriebene SQL eingeordnet wird — eine einzige breite
Tabelle mit allen Metriken × allen drei Varianten wäre nicht mehr lesbar.

In [5]:
def print_metric_table(title, key, decimals=None, unit=""):
    """
    Zeigt eine Metrik für EQL, GenSQL und Hand nebeneinander, je eine Zeile
    pro Query. Erwartet, dass jedes r in `queries` die Keys 'eql', 'sql' und
    'handwritten' mit dem gegebenen `key` enthält.
    """
    print(f"\n--- {title} ---")
    print(f'{"Query":<26} {"EQL":>10} {"GenSQL":>10} {"Hand":>10}')
    print("-" * 60)
    for r in queries:
        eql_v, sql_v, hand_v = r["eql"][key], r["sql"][key], r["handwritten"][key]
        if decimals is not None:
            eql_v, sql_v, hand_v = (round(v, decimals) for v in (eql_v, sql_v, hand_v))
        print(f'{r["name"]:<26} {eql_v:>10}{unit} {sql_v:>10}{unit} {hand_v:>10}{unit}')


print_metric_table("Kognitive Komplexität (CC)", "cognitive_complexity")
print_metric_table("Zeichenlänge (len)", "char_length")
print_metric_table("FK-Fehlermöglichkeiten (FK)", "fk_error_opportunities")

print("\n--- Abstraktionsverhältnis (Zeichen SQL-Variante pro EQL-Zeichen) ---")
print(f'{"Query":<26} {"GenSQL/EQL":>12} {"Hand/EQL":>10}')
print("-" * 50)
for r in queries:
    print(
        f'{r["name"]:<26} '
        f'{r["shared"]["abstraction_ratio"]:>11.1f}x '
        f'{r["shared"]["abstraction_ratio_handwritten"]:>9.1f}x'
    )
print()
print("GenSQL/EQL = Übersetzer-SQL-Zeichen pro EQL-Zeichen;")
print("Hand/EQL   = handgeschriebene SQL-Zeichen pro EQL-Zeichen —")
print("             >1.0 = EQL ist kürzer als das, was ein Mensch schreiben würde;")
print("             <1.0 = handgeschriebenes SQL ist (für diesen Fall) sogar kürzer.")


--- Kognitive Komplexität (CC) ---
Query                             EQL     GenSQL       Hand
------------------------------------------------------------
Simple filter                       3          3          3
Transitive attrs                    1          8          2
Multi-join (3 vars)                 4         18         12
Multi-join (4 vars)                 5         25         16
Complex combined                    3          3          3
PickUpAction-Beispiel               1          4          2

--- Zeichenlänge (len) ---
Query                             EQL     GenSQL       Hand
------------------------------------------------------------
Simple filter                      61        377        150
Transitive attrs                  148        589        134
Multi-join (3 vars)               182       1853        684
Multi-join (4 vars)               254       2632        945
Complex combined                   90        411        154
PickUpAction-Beispiel             

## 5. Zusatz — Übersetzer-Overhead ggü. handgeschriebenem SQL (Kontext, kein EQL-Vergleich)

Die EQL-vs-Hand- und EQL-vs-GenSQL-Vergleiche stehen bereits vollständig in
Tabelle 1. Diese Zusatztabelle beantwortet eine andere, enger gefasste
Frage: Wie nah kommt der *Übersetzer* an das, was ein Mensch für dasselbe
SQL schreiben würde? Das trennt schema-inhärenten JTI-Aufwand von
Übersetzer-Overhead, sagt aber nichts über EQL selbst aus. Wie viel davon
tatsächlich vermeidbar ist, ist nicht abschließend geklärt: Die überflüssig
mitgejointen Vorfahrtabellen sind es sicher; bei der ausführlichen
Spaltenselektion bleibt offen, ob der Übersetzer sie braucht, um die
DAO-Objekte des Ergebnisses zu rekonstruieren (Hand selektiert nur eine
Teilmenge der Spalten).

In [6]:
print(f'{"Query":<26} {"Hand len":>9} {"GenSQL len":>11} {"GenSQL/Hand":>12}')
print('-' * 62)
for r in queries:
    print(
        f'{r["name"]:<26} '
        f'{r["handwritten"]["char_length"]:>9} '
        f'{r["sql"]["char_length"]:>11} '
        f'{r["shared"]["translator_overhead_ratio"]:>11.2f}x'
    )
print()
print('GenSQL/Hand = Übersetzer-SQL-Zeichen pro Zeichen handgeschriebenen SQLs;')
print('              1.0 = Übersetzer trifft genau das, was ein Mensch schreiben würde;')
print('              >1.0 = Übersetzer-Overhead (z. B. unnötig mitgejointe Tabellen);')
print('              nicht in voller Höhe vermeidbar, da Hand weniger Spalten selektiert.')

Query                       Hand len  GenSQL len  GenSQL/Hand
--------------------------------------------------------------
Simple filter                    150         377        2.51x
Transitive attrs                 134         589        4.40x
Multi-join (3 vars)              684        1853        2.71x
Multi-join (4 vars)              945        2632        2.79x
Complex combined                 154         411        2.67x
PickUpAction-Beispiel            132         431        3.27x

GenSQL/Hand = Übersetzer-SQL-Zeichen pro Zeichen handgeschriebenen SQLs;
              1.0 = Übersetzer trifft genau das, was ein Mensch schreiben würde;
              >1.0 = Übersetzer-Overhead (z. B. unnötig mitgejointe Tabellen);
              nicht in voller Höhe vermeidbar, da Hand weniger Spalten selektiert.


## 6. Tabelle 2 — Lesbarkeitsmetriken (symmetrisch gezählt)

Wie in Tabelle 1: jede Metrik als EQL/GenSQL/Hand-Tabelle, damit auch die
Lesbarkeitsmetriken nicht nur gegen das generierte, sondern konsequent auch
gegen handgeschriebenes SQL eingeordnet werden.

In [7]:
print_metric_table("Technischer Lärm (noise)", "technical_noise")
print_metric_table("Domänen-Transparenz (transp)", "domain_transparency", decimals=3)
print_metric_table("Verschachtelungstiefe (nest)", "nesting_depth")
print_metric_table("Halstead-Volumen (Hals)", "halstead_volume", decimals=2)
print_metric_table("Extrinsische kognitive Last (ext)", "extrinsic_load")

print('\nLegende:')
print('  EQL = EQL-Quelltext, GenSQL = Übersetzer-generiertes SQL, Hand = handgeschriebenes SQL')
print('  noise  = Technischer Lärm (Lawrie et al. 2006), symmetrisch gezählt')
print('  transp = Domänen-Transparenz (Fowler 2010), symmetrisch gezählt')
print('  nest   = Verschachtelungstiefe — SQL gewinnt by design (Campbell 2018)')
print('  Hals   = Halstead-Volumen (Halstead 1977)')
print('  ext    = Extrinsische kognitive Last (Sweller 1988)')


--- Technischer Lärm (noise) ---
Query                             EQL     GenSQL       Hand
------------------------------------------------------------
Simple filter                       6         33         15
Transitive attrs                    5         38          8
Multi-join (3 vars)                 9        118         63
Multi-join (4 vars)                11        159         86
Complex combined                    8         37         17
PickUpAction-Beispiel               5         25          8

--- Domänen-Transparenz (transp) ---
Query                             EQL     GenSQL       Hand
------------------------------------------------------------
Simple filter                   0.455      0.154      0.483
Transitive attrs                0.706      0.269        0.6
Multi-join (3 vars)               0.7      0.337      0.427
Multi-join (4 vars)             0.725      0.354      0.419
Complex combined                0.467      0.178      0.452
PickUpAction-Beispiel     

## 7. Skalierung über 5 Datenpunkte (statt 2)

**Wichtig:** Die EQL-Quelltexte enthalten die `variable(...)`-Deklarationen (siehe `eql_source_for_chain`).

**Wichtig:** Diese Ketten (`build_container_chain(k)`) sind synthetisch, nicht
eine der fünf realen Evaluationsqueries — dafür gibt es kein handgeschriebenes
SQL, nur EQL vs. `GenSQL` (Übersetzer-generiert). Die Tabelle zeigt, wie beide
mit wachsender Variablenzahl wachsen, jeweils als Wachstumsfaktor zum
vorherigen Messpunkt (nicht als Verhältnis SQL/EQL pro Zeile — das war in
einer früheren Version missverständlich beschriftet). Zusätzlich werden das
Verhältnis GenSQL/EQL und der absolute Zuwachs in Zeichen pro Schritt gezeigt,
weil das relative Wachstum bei der kürzeren Länge (EQL) rechnerisch höher
ausfällt, auch wenn der absolute Zuwachs ähnlich ist.

In [8]:
scaling_rows = []
for k in range(2, 7):
    world_k = build_container_chain(k)
    C_k = [variable(Container, domain=world_k.bodies) for _ in range(k)]
    FC_k = [variable(FixedConnection, domain=world_k.connections) for _ in range(k - 1)]

    conditions = []
    for i in range(k - 1):
        conditions.append(C_k[i] == FC_k[i].parent)
        conditions.append(C_k[i + 1] == FC_k[i].child)

    all_vars = list(C_k) + list(FC_k)
    query_k = an(set_of(*all_vars).where(*conditions))
    translator_k = eql_to_sql(query_k, session)

    eql_src_k = m.eql_source_for_chain(k)
    sql_src_k = str(translator_k.sql_query)
    scaling_rows.append({
        "num_vars": 2 * k - 1,
        "eql_len": m.character_length(eql_src_k),
        "sql_len": m.character_length(sql_src_k),
    })

print(f'{"Vars":>5} {"EQL len":>9} {"GenSQL len":>11} {"GenSQL/EQL":>11} '
      f'{"EQL-Wachstum":>13} {"GenSQL-Wachstum":>16} {"EQL +Zeichen":>13} {"GenSQL +Zeichen":>16}')
print("-" * 104)
prev_eql, prev_sql = None, None
for row in scaling_rows:
    ratio = f"{round(row['sql_len']/row['eql_len'], 1)}x"
    eql_growth = f"{round(row['eql_len']/prev_eql, 2)}x" if prev_eql else ""
    sql_growth = f"{round(row['sql_len']/prev_sql, 2)}x" if prev_sql else ""
    eql_abs = row['eql_len'] - prev_eql if prev_eql else ""
    sql_abs = row['sql_len'] - prev_sql if prev_sql else ""
    print(
        f'{row["num_vars"]:>5} {row["eql_len"]:>9} {row["sql_len"]:>11} {ratio:>11} '
        f'{eql_growth:>13} {sql_growth:>16} {eql_abs:>13} {sql_abs:>16}'
    )
    prev_eql, prev_sql = row["eql_len"], row["sql_len"]

print()
print(
    "EQL-Wachstum / GenSQL-Wachstum = Längenverhältnis zum jeweils vorherigen"
    " Messpunkt; '+Zeichen' = absoluter Zuwachs pro Schritt. Beides ist ein"
    " Trend über fünf Messungen an einer synthetischen Kette, keine Aussage"
    " über eine Komplexitätsklasse und keine Extrapolation auf andere Queries."
)

assert len(scaling_rows) == 5
assert scaling_rows[-1]['sql_len'] > scaling_rows[0]['sql_len']
assert scaling_rows[-1]['eql_len'] > scaling_rows[0]['eql_len']
print("\nRegressions-Check bestanden: beide Längen steigen über die 5 Punkte.")


 Vars   EQL len  GenSQL len  GenSQL/EQL  EQL-Wachstum  GenSQL-Wachstum  EQL +Zeichen  GenSQL +Zeichen
--------------------------------------------------------------------------------------------------------
    3       195        1569        8.0x                                                              
    5       327        1690        5.2x         1.68x            1.08x           132              121
    7       459        1809        3.9x          1.4x            1.07x           132              119
    9       591        1928        3.3x         1.29x            1.07x           132              119
   11       723        2047        2.8x         1.22x            1.06x           132              119

EQL-Wachstum / GenSQL-Wachstum = Längenverhältnis zum jeweils vorherigen Messpunkt; '+Zeichen' = absoluter Zuwachs pro Schritt. Beides ist ein Trend über fünf Messungen an einer synthetischen Kette, keine Aussage über eine Komplexitätsklasse und keine Extrapolation auf andere Queri

/tmp/ipykernel_154/883368449.py:17: SAWarning: An alias is being generated automatically against joined entity Mapper[FixedConnectionDAO(FixedConnectionDAO)] due to overlapping tables.  This is a legacy pattern which may be deprecated in a later release.  Use the aliased(<entity>, flat=True) construct explicitly, see the linked example. (Background on this warning at: https://sqlalche.me/e/20/xaj2)
  sql_src_k = str(translator_k.sql_query)
/tmp/ipykernel_154/883368449.py:17: SAWarning: An alias is being generated automatically against joined entity Mapper[ContainerDAO(ContainerDAO)] due to overlapping tables.  This is a legacy pattern which may be deprecated in a later release.  Use the aliased(<entity>, flat=True) construct explicitly, see the linked example. (Background on this warning at: https://sqlalche.me/e/20/xaj2)
  sql_src_k = str(translator_k.sql_query)


## 8. Feature-Tabelle statt bloßer Prozentzahl

In [9]:
m.print_feature_table()

EQL feature coverage (explicit, traceable):
Feature                                                     Status
------------------------------------------------------------------
SELECT specific columns (set_of)                               yes
SELECT full entity (entity)                                    yes
WHERE / filter conditions                                      yes
Equality joins across variables                                yes
Transitive attribute access (a.b.c)                            yes
ORDER BY (ordered_by)                                          yes
LIMIT                                                          yes
DISTINCT                                                       yes
AND / OR (and_, or_)                                           yes
NOT / IN (not_, in_)                                           yes
GROUP BY (grouped_by)                                          yes
HAVING (having)                                                yes
CASE WHEN (case_wh

## 9. Symmetrische Domain-Transparency am PickUpAction-Beispiel

In [10]:
tabelle_row = next(r for r in queries if r['name'] == 'PickUpAction-Beispiel')

naive_eql = m.domain_transparency_naive(m.PICKUP_EXAMPLE_EQL)
naive_sql = m.domain_transparency_naive(m.PICKUP_EXAMPLE_TRANSLATOR_SQL)
honest_eql = m.domain_transparency(m.PICKUP_EXAMPLE_EQL, True)
honest_sql = m.domain_transparency(m.PICKUP_EXAMPLE_TRANSLATOR_SQL, False)

print(f'{"":8} {"naiv":>8} {"symmetrisch":>13}')
print(f'{"EQL":<8} {naive_eql:>8.3f} {honest_eql:>13.3f}')
print(f'{"GenSQL":<8} {naive_sql:>8.3f} {honest_sql:>13.3f}')

assert honest_eql < naive_eql, "Symmetrisches Zählen sollte den naiven EQL-Wert senken"
assert honest_eql > honest_sql, "EQL sollte auch symmetrisch gezählt vor SQL liegen"
print("\nRegressions-Check bestanden.")

print(
    f"\nÜbersetzer-Overhead ggü. handgeschrieben (PickUpAction-Beispiel): "
    f"{tabelle_row['shared']['translator_overhead_ratio']}x"
)
print(
    f"Abstraction ratio EQL vs. handgeschrieben (PickUpAction-Beispiel): "
    f"{tabelle_row['shared']['abstraction_ratio_handwritten']}x "
    f"({'EQL kürzer' if tabelle_row['shared']['abstraction_ratio_handwritten'] > 1 else 'handgeschrieben kürzer oder gleich lang'})"
)

             naiv   symmetrisch
EQL         1.000         0.688
GenSQL      0.094         0.219

Regressions-Check bestanden.

Übersetzer-Overhead ggü. handgeschrieben (PickUpAction-Beispiel): 3.27x
Abstraction ratio EQL vs. handgeschrieben (PickUpAction-Beispiel): 0.8x (handgeschrieben kürzer oder gleich lang)


## Zusammenfassung

Alle Tabellen und Zahlen oben können direkt zur Aktualisierung der
entsprechenden Thesis-Tabellen und der Diskussion des PickUpAction-Beispiels
in der Einleitung verwendet werden. Anders als die alten Zahlen sind sie:

- symmetrisch gezählt (EQLs eigene DSL-Wörter zählen genauso als
  Infrastruktur wie SQL-Schlüsselwörter),
- gegen realistisch handgeschriebenes SQL verglichen, nicht nur gegen das
  Übersetzer-SQL,
- über 5 statt 2 Punkten skaliert,
- mit einer nachvollziehbaren Feature-Tabelle statt einer bloßen
  Prozentzahl,
- und zeigen Unentschieden explizit statt sie zu verschweigen.